# OpenPlaque RCA Centerline Validation

Research prototype for validating the proximal RCA centerline needed for standardized PCAT sampling. This notebook expects a CCTA volume and an **isolated RCA mask** in NIfTI format. It extracts the centerline, reports path length, and marks 0, 10, and 50 mm from the supplied ostium.

Do not use the 10–50 mm region for quantitative PCAT analysis until the centerline and landmarks have been visually checked.

In [ ]:
!pip -q install git+https://github.com/pazzani/OpenPlaque.git@rca-centerline-prototype
import numpy as np
import SimpleITK as sitk
import matplotlib.pyplot as plt
from openplaque.centerline import extract_rca_centerline, show_centerline_mip


## 1. Configure one case
Set paths to the original CCTA and an isolated RCA mask. Coordinates below use NumPy order **(z, y, x)**. The distal hint is strongly recommended when the RCA mask contains branches.

In [ ]:
CT_PATH = '/content/ccta.nii.gz'
RCA_MASK_PATH = '/content/rca_mask.nii.gz'

# Approximate points; the algorithm snaps them to the nearest skeleton voxel.
OSTIUM_ZYX = (0, 0, 0)       # EDIT
DISTAL_HINT_ZYX = None        # e.g. (120, 180, 210); strongly recommended

# If the mask is already binary, leave this as None. Otherwise set the RCA foreground label.
RCA_LABEL = None


In [ ]:
ct_img = sitk.ReadImage(CT_PATH)
mask_img = sitk.ReadImage(RCA_MASK_PATH)
ct = sitk.GetArrayFromImage(ct_img)
mask_raw = sitk.GetArrayFromImage(mask_img)
spacing_xyz = ct_img.GetSpacing()

if ct.shape != mask_raw.shape:
    raise ValueError(f'CT and RCA mask shapes differ: {ct.shape} vs {mask_raw.shape}')

rca = mask_raw.astype(bool) if RCA_LABEL is None else (mask_raw == RCA_LABEL)
print('CT shape:', ct.shape)
print('Spacing xyz (mm):', spacing_xyz)
print('RCA voxels:', int(rca.sum()))


## 2. Sanity-check the ostium slice
The supplied point only needs to be near the proximal RCA. Check that it is on the intended vessel before extracting the path.

In [ ]:
z = int(OSTIUM_ZYX[0])
fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(ct[z], cmap='gray', vmin=-200, vmax=800)
ax.contour(rca[z].astype(float), levels=[0.5], linewidths=1.2)
ax.scatter([OSTIUM_ZYX[2]], [OSTIUM_ZYX[1]], s=70, marker='x')
ax.set_title(f'RCA mask and proposed ostium, z={z}')
ax.axis('off');


## 3. Extract the centerline and landmarks

In [ ]:
result = extract_rca_centerline(
    rca,
    spacing_xyz,
    OSTIUM_ZYX,
    distal_hint_zyx=DISTAL_HINT_ZYX,
    landmark_distances_mm=(0.0, 10.0, 50.0),
)

print(f'Centerline length: {result.length_mm:.1f} mm')
print('Snapped ostium zyx:', result.ostium_zyx_voxel)
print('Endpoint zyx:', result.endpoint_zyx_voxel)
for d, p in sorted(result.landmarks_xyz_mm.items()):
    print(f'{d:>4.0f} mm -> xyz mm {np.round(p, 2)}')
if 50.0 not in result.landmarks_xyz_mm:
    print('WARNING: extracted path is shorter than 50 mm; standardized 10–50 mm RCA region is unavailable.')


## 4. Inspect all three projections
A valid path should stay centered within the intended RCA, avoid branch jumps, and place the 10 and 50 mm landmarks on the same main RCA path.

In [ ]:
for axis, name in [(0, 'axial MIP'), (1, 'coronal MIP'), (2, 'sagittal MIP')]:
    fig, ax = show_centerline_mip(ct, result, axis=axis)
    ax.set_title(f'{name}: RCA centerline, length {result.length_mm:.1f} mm')
    plt.show()


## 5. Quantitative QC
These are simple geometry checks, not a validation against an expert reference. The key first-pass criterion is still visual continuity on the intended RCA.

In [ ]:
pts = result.points_xyz_mm
seg = np.linalg.norm(np.diff(pts, axis=0), axis=1) if len(pts) > 1 else np.array([])
direct = np.linalg.norm(pts[-1] - pts[0]) if len(pts) > 1 else 0.0
tortuosity = result.length_mm / direct if direct > 0 else np.nan
print(f'Number of centerline voxels: {len(pts)}')
print(f'Median step length: {np.median(seg):.3f} mm' if len(seg) else 'Median step length: n/a')
print(f'Max step length: {np.max(seg):.3f} mm' if len(seg) else 'Max step length: n/a')
print(f'Path/direct-distance ratio: {tortuosity:.3f}')
print('Landmarks available:', sorted(result.landmarks_xyz_mm))


### Pass/fail record
For each case record: (1) correct RCA component, (2) no branch jump, (3) centerline visually inside/near lumen center, (4) 10 mm landmark plausible, (5) 50 mm landmark plausible. Once this passes on a representative set, the next module can sample PCAT from 10–50 mm using the vessel/outer-wall geometry rather than the centerline radius itself.